# Lesson 8 — Unsupervised Learning

Self-assessment. No code: every answer is a sentence, a short derivation, or a
diagnosis.

Numbers quoted throughout come from the lesson's handout and notebooks:
2,000 Aurora customers segmented by annual spend and visit frequency into 4
true segments; 1,500 browsing sessions, 12% of them bots by construction;
2,000 accounts described by 8 correlated columns, generated from 3 latent
factors, with 40 planted fraudulent accounts. As in earlier lessons, several
questions ask you to *derive* or *criticise* a result rather than recall it;
those are the ones worth your time.

## Part 1 — k-means: the objective and Lloyd's algorithm

**1. Contrast supervised and unsupervised learning in one sentence, and state the k-means objective <code>J</code> precisely, including what <code>r<sub>ij</sub></code> means.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Supervised learning learns from pairs (x, y) &mdash; an input and a target to predict; unsupervised learning removes y entirely and instead describes structure in the inputs alone: which points look alike, which few numbers summarise the many, which points do not belong.</li>
        <li>k-means seeks the assignment of every point x<sub>i</sub> to one of k clusters and the k cluster centres &mu;<sub>1</sub>, &hellip;, &mu;<sub>k</sub> that jointly minimise the <b>within-cluster sum of squares</b> (WCSS): <code>J = &Sigma;<sub>i</sub> &Sigma;<sub>j</sub> r<sub>ij</sub> &lVert;x<sub>i</sub> &minus; &mu;<sub>j</sub>&rVert;&sup2;</code>.</li>
        <li><code>r<sub>ij</sub> &isin; {0, 1}</code> is 1 exactly when point i is assigned to cluster j, with <code>&Sigma;<sub>j</sub> r<sub>ij</sub> = 1</code> for every point &mdash; each point belongs to exactly one cluster.</li>
    </ul>
    </p>
</details>

**2. Derive the exact minimiser of J for each half of the k-means problem &mdash; the assignment step with the centres fixed, and the update step with the assignments fixed &mdash; and explain why minimising over both simultaneously is hard.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Minimising over both kinds of variable at once is computationally hard because there are more ways to partition m points into k groups than any search can enumerate once m passes a few dozen &mdash; there is no shortcut to trying every partition.</li>
        <li><b>Fixing the centres, minimise over the assignment:</b> for each point i independently, J is a sum over j of the non-negative term r<sub>ij</sub>&lVert;x<sub>i</sub>&minus;&mu;<sub>j</sub>&rVert;&sup2; with exactly one r<sub>ij</sub> allowed to be 1. The minimum sets r<sub>ij</sub>=1 for whichever j minimises &lVert;x<sub>i</sub>&minus;&mu;<sub>j</sub>&rVert;&sup2; &mdash; assign every point to its <b>nearest centre</b>.</li>
        <li><b>Fixing the assignment, minimise over the centres:</b> for each cluster j independently, J restricted to cluster j's points is a convex quadratic in &mu;<sub>j</sub>. Setting its gradient to zero, &minus;2&Sigma;(x<sub>i</sub>&minus;&mu;<sub>j</sub>)=0, gives &mu;<sub>j</sub> equal to the <b>mean</b> of the points currently assigned to cluster j.</li>
        <li>Neither minimisation alone solves the joint problem, but alternating the two exact minimisations &mdash; Lloyd's algorithm &mdash; is what makes k-means tractable at all.</li>
    </ul>
    </p>
</details>

**3. Explain precisely why Lloyd's algorithm is guaranteed to converge, and illustrate with the six-point worked example, whose WCSS ran 15.965 &rarr; 7.281 &rarr; 5.491 &rarr; 4.674.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Each step of Lloyd's algorithm &mdash; assign, then update &mdash; is an exact minimisation of J over the variables it controls, so it can only decrease J or leave it unchanged, never increase it.</li>
        <li>J is bounded below by 0, and there are only finitely many ways to partition m points into k groups, so the non-increasing sequence of J values cannot decrease forever: it must eventually reach a partition that repeats, at which point neither step changes anything and the algorithm has converged.</li>
        <li>In the six-point example, J falls at every step (15.965 &rarr; 7.281 &rarr; 5.491 &rarr; 4.674) and by iteration 2 the assignment repeats with both steps leaving J unchanged &mdash; convergence, and in this case to the global optimum for these six points. What Lloyd's algorithm converges to <i>in general</i> is only a <b>local</b> minimum, not necessarily the global one.</li>
    </ul>
    </p>
</details>

## Part 2 — Initialisation and choosing k

**4. Explain why thirty independent naive-random-start runs of Lloyd's algorithm on the 2,000-customer data gave a best final WCSS of 374.1 and a worst of 1,390.4, and how k-means++ initialisation reduces that risk.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Naive random initialisation picks k points uniformly at random from the data as starting centres; which local minimum Lloyd's algorithm reaches depends entirely on where those centres start, with no way to tell in advance whether a given start is good.</li>
        <li>On this data, the worst of the 30 starts left the algorithm stuck in a local minimum <b>3.7 times worse</b> than the best (1,390.4 vs 374.1) using the identical update rule on identical data &mdash; and this happened on the very first attempt, not a rare pathology.</li>
        <li><b>k-means++</b> addresses this at initialisation: the first centre is uniformly random, and every subsequent centre is chosen with probability proportional to its squared distance from the nearest centre already chosen, so a point far from every existing centre &mdash; plausibly an unrepresented cluster's seed &mdash; is favoured. Across the same 30 seeds this narrowed the worst case to 1,088.3 and reached the global optimum on the very first attempt with default settings.</li>
    </ul>
    </p>
</details>

**5. Compare the elbow method and the silhouette score as diagnostics for choosing k: what does each measure, and why is checking both worth the small extra cost?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The <b>elbow</b> looks at WCSS as a function of k. WCSS is monotonically non-increasing in k (at k=m it is exactly 0), so the raw value at the best k is not the signal &mdash; what matters is where the curve stops falling steeply, since past the true number of clusters an extra centre only subdivides an already-coherent cluster.</li>
        <li>The <b>silhouette score</b>, for point i, compares its mean distance to its own cluster (a<sub>i</sub>) against its mean distance to the nearest other cluster (b<sub>i</sub>): <code>s<sub>i</sub> = (b<sub>i</sub> &minus; a<sub>i</sub>) / max(a<sub>i</sub>, b<sub>i</sub>) &isin; [&minus;1, 1]</code>. Averaged over all points it is a genuine optimum to search over, not only a shape to eyeball.</li>
        <li>The two measure different things &mdash; WCSS asks only whether adding a cluster helps, agnostic to what a cluster <i>means</i>; silhouette asks whether every point is closer to its own group than to any rival, a stronger geometric condition &mdash; so agreement between them is informative precisely because it is not guaranteed.</li>
    </ul>
    </p>
</details>

**6. State what a silhouette value near 1 means for a single point and what a value below 0 means, then say which k both diagnostics agree on for the customer-segment data (silhouette peaks at 0.690 at k=4) and why that agreement is not automatic.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>s<sub>i</sub> &rarr; 1 when a point sits deep inside its own cluster and far from every other cluster; s<sub>i</sub> &lt; 0 means the point is, on average, <b>closer to a different cluster than to its own</b> &mdash; an assignment the silhouette treats as actively wrong, not merely ambiguous.</li>
        <li>Both diagnostics point to k=4 on this dataset: WCSS falls sharply through k=4 (374.1) and flattens after (only down to 320.6 at k=5), and silhouette peaks at exactly k=4 (0.690) before declining to 0.593 at k=5.</li>
        <li>The agreement is not automatic &mdash; it is a property of this dataset's four well-separated, comparably sized blobs (and it happens to match the number of segments the data was generated with), not a guarantee that holds for every dataset; section 4 of the handout warns explicitly against reading it as one.</li>
    </ul>
    </p>
</details>

## Part 3 — When round clusters are the wrong assumption

**7. Describe the actual shape of the bot and human groups in the session data &mdash; built so both share the same mean duration and page count &mdash; and explain geometrically why k-means with k=2 fails on it (ARI = &minus;0.046).**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Bots (12% of 1,500 sessions) form a tight cloud centred at (4.0 min, 12 pages); humans (88%) are spread into a ring <i>around that same centre</i>, not a separate blob beside it &mdash; there is no offset between the two groups' means to exploit.</li>
        <li>k-means minimises WCSS, which is minimised by round, compact clusters; the only way to cut this cloud into two round pieces is a line straight through the centre, which slices <i>both</i> the bot core and the human ring in half rather than separating them.</li>
        <li>The result, ARI = &minus;0.046, is statistically indistinguishable from a random split of the same two group sizes, and no choice of initialisation fixes it &mdash; the failure is in what k-means optimises, not in how it was started.</li>
    </ul>
    </p>
</details>

**8. Define the four linkage criteria used by agglomerative (hierarchical) clustering to measure the distance between two clusters.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Single linkage:</b> the distance between the closest pair of points, one from each cluster.</li>
        <li><b>Complete linkage:</b> the distance between the farthest pair of points, one from each cluster.</li>
        <li><b>Average linkage:</b> the mean distance over every cross-pair between the two clusters.</li>
        <li><b>Ward linkage:</b> merges whichever pair of clusters increases total within-cluster variance the least &mdash; the same objective k-means minimises, expressed as a greedy merge rule rather than an iterative update.</li>
    </ul>
    </p>
</details>

**9. Explain the mechanism (chaining) that made single and average linkage each cut the bot/human data into a cluster of 1,499 points and a cluster of 1, and why it happens for these two linkage rules specifically.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Single linkage only needs <i>one</i> close pair of points to justify a merge, so it can trace a thin, winding chain of points arbitrarily far apart end to end &mdash; and just as easily string together two otherwise-separate groups that touch at a single bridging point, a failure called <b>chaining</b>. Average linkage, being a mean over cross-pairs, is less extreme but shares the same vulnerability when the data is one continuously dense mass.</li>
        <li>On the bot/human ring-and-core data there is no true gap for either rule to detect &mdash; the ring is dense enough, and close enough to the core, that both rules absorb almost the entire dataset along its densest path before running out of points to merge.</li>
        <li>The result &mdash; 1,499 points in one cluster and 1 in the other, for both linkages &mdash; is not a discovery of structure; it is the visible signature of chaining, and it leaves nothing resembling the actual bot/human split (ARI &asymp; &minus;0.001 for both).</li>
    </ul>
    </p>
</details>

**10. Ward and complete linkage also fail on the bot/human data (ARI &minus;0.062 and &minus;0.101). Having just watched k-means recover the customer segments almost perfectly in section 2, why is it a mistake to conclude that clustering as a category has now been shown to work well?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The instinct is not wrong about k-means specifically &mdash; k-means genuinely did recover the customer segments well. It is wrong about generalising from that to clustering as a whole, because different methods optimise different notions of &ldquo;cluster.&rdquo;</li>
        <li>k-means and Ward linkage both optimise for compact, round clusters; the bot/human data has no such structure &mdash; it has a dense core inside a diffuse ring, a shape neither method's objective can represent, regardless of how well it is tuned or initialised.</li>
        <li>Which definition of &ldquo;cluster&rdquo; is the right one is a property of the data, not of the algorithm &mdash; which is why looking at the unlabelled scatter plot, before running anything, is the step that actually catches this, not a higher score from a different method.</li>
    </ul>
    </p>
</details>

## Part 4 — DBSCAN: density instead of shape

**11. Define DBSCAN's (density-based spatial clustering of applications with noise) three point types &mdash; core, border, and noise &mdash; in terms of its two parameters <code>eps</code> and <code>min_samples</code>.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A <b>core point</b> has at least <code>min_samples</code> other points within distance <code>eps</code> of it.</li>
        <li>A <b>border point</b> is not itself a core point, but lies within <code>eps</code> of one &mdash; it belongs to that core's cluster but cannot extend it further.</li>
        <li>Every other point is <b>noise</b>. A cluster is a maximal set of core points connected by chains of mutual eps-closeness, together with every border point attached to them &mdash; nothing in this definition mentions a centre, a mean, or a shape.</li>
    </ul>
    </p>
</details>

**12. Explain what a <code>k</code>-distance plot shows and how it is used to choose <code>eps</code>, and describe what goes wrong if <code>eps</code> is chosen too small or too large.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A k-distance plot sorts, from smallest to largest, the distance from every point to its <code>min_samples</code>-th nearest neighbour: points in dense regions have a small such distance, points in sparse regions a large one, and the plot typically shows a sharp bend between the two regimes &mdash; a principled place to set <code>eps</code>.</li>
        <li>Too <b>small</b> an <code>eps</code>, and even the genuinely dense bot core fractures into isolated noise points, since fewer than <code>min_samples</code> neighbours fall within reach of any one point.</li>
        <li>Too <b>large</b> an <code>eps</code>, and the ring's outer reaches bridge back across the gap into the core, merging everything into one cluster &mdash; exactly the failure Ward linkage had, reached by a different mechanism.</li>
    </ul>
    </p>
</details>

**13. Explain why DBSCAN calling 13 sessions &ldquo;noise&rdquo; &mdash; all 13 turn out to be genuine humans &mdash; is a more honest answer than forcing every point into one of two clusters, given that eps = 0.30, min_samples = 10 reaches ARI = 0.941 overall.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Every one of the 180 bot sessions lands in DBSCAN's dense-core cluster and 1,303 of the 1,320 human sessions land in the other cluster; only 4 humans are grouped with the bots, and 13 are called noise rather than being forced into either group.</li>
        <li>Those 13 are genuine human sessions whose one week of browsing happened, by chance, to look almost as mechanically regular as a script's &mdash; points that are not confidently either group.</li>
        <li>k-means and every linkage rule in section 3.2 must assign <i>every</i> point to exactly one cluster; DBSCAN's &ldquo;noise&rdquo; label is an answer none of those methods is even capable of giving, and here it is the correct answer for those 13 points rather than an evasion.</li>
    </ul>
    </p>
</details>

**14. A colleague has a dataset where the number of natural groups is unknown, the groups may be irregular or of very different densities, and any genuine outliers should be flagged rather than forced into a group. Which of the three clustering methods in this lesson should they reach for, and why would the other two be a worse fit?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Reach for <b>DBSCAN</b>: it clusters by density rather than shape, discovers the number of clusters itself rather than requiring k as an input, and has a built-in mechanism &mdash; the noise label &mdash; for points that do not belong confidently anywhere.</li>
        <li><b>k-means</b> would be a poor fit: it requires k in advance, and its WCSS objective assumes clusters are round and comparably sized, which &ldquo;irregular, of very different densities&rdquo; explicitly rules out.</li>
        <li><b>Agglomerative clustering</b> with Ward or complete linkage shares k-means' bias toward compact, round clusters; single or average linkage avoids that bias but is prone to chaining across exactly the kind of irregular density variation described here.</li>
    </ul>
    </p>
</details>

## Part 5 — Validating a clustering with no labels to check against

**15. Distinguish internal from external cluster-validation metrics, name one example of each used in this lesson, and state why external metrics are not always available in practice.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Internal metrics</b> use only the clustering and the data, never an outside answer key &mdash; the silhouette score is one, since it asks whether points are closer to their own cluster than to any other, a question answerable from the clustering alone.</li>
        <li><b>External metrics</b> &mdash; the adjusted Rand index (ARI) is this lesson's example &mdash; compare a clustering against a separate labelling assumed to be correct, corrected so a random assignment scores 0 and identical labellings score 1.</li>
        <li>In this lesson the separate labelling exists only because the data is synthetic (<code>true_segment</code>, <code>true_group</code>); real problems rarely have that. In practice an external signal might be a small hand-labelled sample, a business rule, or a downstream outcome &mdash; and when none of these exists, only internal metrics are available at all.</li>
    </ul>
    </p>
</details>

**16. Explain precisely what a silhouette score of 0.69, reported without qualification, is and is not evidence of, using k-means on the bot/human data (ARI &asymp; &minus;0.046 despite an unremarkable-looking silhouette) as the concrete case.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A silhouette score is only ever an endorsement <i>relative to what the metric can see</i> &mdash; the shape of the clusters it was given &mdash; not of whether the clustering algorithm's underlying assumption fits the data at all.</li>
        <li>k-means on the bot/human data produces round-looking clusters by construction, so its silhouette score there comes out positive and unremarkable-looking, even though its ARI against the true bot/human split is effectively zero.</li>
        <li>Silhouette is measuring roundness (internal self-consistency), not correctness against any outside truth &mdash; an internal metric has no way to know that k-means' round-cluster assumption is the wrong one for a given dataset.</li>
    </ul>
    </p>
</details>

## Part 6 — Principal component analysis, derived two ways

**17. Explain, without equations, what principal component analysis (PCA) is trying to find when it looks for the &ldquo;first principal component,&rdquo; and why &ldquo;varies the most&rdquo; is used as a proxy for &ldquo;carries the most information.&rdquo;**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>PCA looks for the direction in feature space along which the data varies the most, calls it the first principal component, then finds the next-most-varying direction perpendicular to the first, and so on.</li>
        <li>A direction along which every point looks nearly identical contributes almost nothing to telling points apart &mdash; it carries little information for distinguishing one point from another &mdash; so it is a safe direction to discard, while a direction of large spread is exactly where points differ from each other.</li>
        <li>Aurora's 8 account columns are not 8 independent pieces of information (spend, basket value and mobile session count all move together); PCA is looking for the handful of directions that actually carry the variation, rather than treating every raw column as equally informative.</li>
    </ul>
    </p>
</details>

**18. Derive why the principal components are the eigenvectors of the covariance matrix &Sigma;, starting from <code>Var(Xv) = v<sup>T</sup>&Sigma;v</code> subject to <code>&lVert;v&rVert; = 1</code>, and use the worked 2&times;2 example (off-diagonal 0.171) to compute the two eigenvalues.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Maximising v<sup>T</sup>&Sigma;v needs the unit-norm constraint &mdash; unconstrained, it grows without bound as v scales up &mdash; so introduce a Lagrange multiplier: <code>L(v,&lambda;) = v<sup>T</sup>&Sigma;v &minus; &lambda;(v<sup>T</sup>v &minus; 1)</code>. Setting <code>&part;L/&part;v = 2&Sigma;v &minus; 2&lambda;v = 0</code> gives <code>&Sigma;v = &lambda;v</code> &mdash; the stationary points are exactly the <b>eigenvectors of &Sigma;</b>, with &lambda; the eigenvalue.</li>
        <li>Substituting back, <code>v<sup>T</sup>&Sigma;v = v<sup>T</sup>(&lambda;v) = &lambda;&lVert;v&rVert;&sup2; = &lambda;</code>: the variance captured by direction v <i>is</i> its eigenvalue. The direction maximising variance is therefore the eigenvector with the <b>largest</b> eigenvalue; the second component is the next-largest eigenvector, automatically orthogonal to the first since &Sigma; is symmetric.</li>
        <li>For any 2&times;2 matrix of the form [[1,r],[r,1]], the eigenvalues are 1+r and 1&minus;r with eigenvectors (1,1)/&radic;2 and (1,&minus;1)/&radic;2. Substituting r = 0.171 gives eigenvalues <b>1.171</b> and <b>0.829</b>, matching <code>numpy.linalg.eigh</code>'s output on the spend/visit-frequency data to three decimal places.</li>
    </ul>
    </p>
</details>

**19. Explain why the singular value decomposition (SVD) of the centred data matrix is numerically preferred over forming &Sigma; = X<sup>T</sup>X/(m&minus;1) explicitly and eigendecomposing it.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The SVD, <code>X = USV<sup>T</sup></code>, never explicitly computes X<sup>T</sup>X; squaring a matrix squares its condition number, the same ill-conditioning concern lesson 3 raised for the normal equation in linear regression.</li>
        <li>The columns of V are exactly the eigenvectors of X<sup>T</sup>X, and <code>X<sup>T</sup>X = VS&sup2;V<sup>T</sup></code>, so <code>&lambda;<sub>i</sub> = s<sub>i</sub>&sup2;/(m&minus;1)</code> recovers the eigenvalues directly from the singular values, without ever forming &Sigma;.</li>
        <li>On the 8-feature account table, eigendecomposition of &Sigma;, SVD of the centred data, and scikit-learn's <code>PCA</code> all agree to within 5&times;10<sup>&minus;15</sup> &mdash; floating-point rounding and nothing more &mdash; confirming the two routes are mathematically identical, with the SVD route simply the more numerically robust way to reach the same answer.</li>
    </ul>
    </p>
</details>

**20. The scree plot shows PC1&ndash;PC3 explaining 44.2%, 32.1%, and 17.2% of variance respectively (93.5% cumulative), with PC4 adding only 2.4% more. What does the sharp elbow at 3 reveal about how the account data was generated, and how general is this pattern likely to be on real data?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The elbow at exactly 3 recovers the number of latent factors (<code>spending_propensity</code>, <code>engagement</code>, <code>price_sensitivity</code>) that actually generated all 8 observed columns as noisy linear combinations &mdash; a number withheld from every method until PCA found it directly from the data.</li>
        <li>This clean an elbow is not guaranteed on real data &mdash; it exists here specifically because the generator built the eight columns from exactly three underlying signals plus noise; real datasets usually show a softer, more gradual decline.</li>
        <li>The general recipe when the elbow is soft is to keep enough components to explain a chosen fraction of variance (commonly 90&ndash;95%), or to treat the number of components as a cross-validated preprocessing hyperparameter and let downstream performance decide.</li>
    </ul>
    </p>
</details>

## Part 7 — Reconstruction error and the anomalies a 2-D plot cannot show

**21. Define the reconstruction <code>x&#770;<sub>i</sub> = V<sub>k</sub>V<sub>k</sub><sup>T</sup>x<sub>i</sub></code> and the reconstruction error <code>e<sub>i</sub></code>, and explain why a genuine account reconstructs well while a planted anomaly does not.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The reconstruction x&#770;<sub>i</sub> projects x<sub>i</sub> onto the k-dimensional subspace spanned by the first k principal components and maps it back &mdash; the closest point to x<sub>i</sub> that lies on that subspace. The reconstruction error is <code>e<sub>i</sub> = &lVert;x<sub>i</sub> &minus; x&#770;<sub>i</sub>&rVert;&sup2;</code>.</li>
        <li>A genuine account's eight values follow the correlation structure the top k components were fit to, so it reconstructs almost exactly &mdash; that structure is exactly what those components describe.</li>
        <li>An anomalous account has individually ordinary values that do not co-occur the way genuine points' do; a subspace built for a correlation pattern it does not follow cannot approximate it well, so its reconstruction error is large &mdash; mean 4.676 for anomalies against 0.431 for genuine accounts at k=3.</li>
    </ul>
    </p>
</details>

**22. State the recall and precision implied by flagging the top 2% of accounts by reconstruction error at k=3 components (threshold 2.019, 40 flagged, 37 of the 40 planted anomalies caught), and interpret what the remaining 3 missed anomalies and 3 false positives mean in practice.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Recall and precision are both <b>92.5%</b>: 37 of the 40 planted anomalies were caught (recall), and 37 of the 40 flagged accounts were genuine anomalies (precision) &mdash; the two coincide here because the same count, 40, appears on both sides of the fraction.</li>
        <li>The 3 missed anomalies happened, by chance, to land close enough to the genuine correlation structure to fall under the threshold; the 3 false positives are genuine accounts that happened to sit above it purely by chance.</li>
        <li>This is a detection problem with a real, quantified error rate, not a guarantee &mdash; it should be reported to Aurora's fraud team framed exactly that way, not as a method that &ldquo;catches fraud,&rdquo; full stop.</li>
    </ul>
    </p>
</details>

**23. Explain the mechanism that makes anomalies sit, on average, closer to the centre than genuine accounts in both a PCA (1.30 vs 2.20) and a t-SNE (22.16 vs 30.17) 2-D projection, and why reconstruction error, not a 2-D scatterplot, is the right tool for this anomaly-detection task.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A genuine account's eight values move together &mdash; high spend tends to come with high basket value and more mobile sessions &mdash; and that co-movement is exactly what pushes a point a meaningful distance along a real principal direction, since the direction of greatest variance <i>is</i> the direction that correlated features jointly vary along.</li>
        <li>An anomalous account's eight values are drawn independently, each from a plausible individual range, with no such co-movement; averaged over eight uncorrelated draws, an anomaly is more likely to land somewhere unremarkable in <i>any single</i> projection than to land consistently far out along one particular direction &mdash; so it ends up looking, on average, more central rather than more extreme.</li>
        <li>The information that identifies these accounts is &ldquo;how well do the kept components predict the columns that were dropped,&rdquo; a question about the relationship between all 8 columns and the discarded components &mdash; a comparison a 2-D scatter of only the kept components has already thrown away, no matter how carefully it is looked at. Reconstruction error uses the entire kept subspace and, implicitly, the discrepancy against what was dropped; a 2-D plot structurally cannot ask that question.</li>
    </ul>
    </p>
</details>

## Part 8 — t-SNE, and choosing among today's methods

**24. State what t-SNE (t-distributed stochastic neighbour embedding) optimises for, and explain the two properties a PCA projection has that a t-SNE embedding does not.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>t-SNE arranges points in 2 or 3 dimensions so that points close together in the original space stay close together in the picture, converting distances into probabilities of being &ldquo;neighbours&rdquo; in both the original and embedded space and moving embedded points to match the two sets of probabilities as closely as possible &mdash; with no commitment to preserving distances between points that were originally far apart.</li>
        <li><b>Distances between clusters in a t-SNE plot are not meaningful</b> &mdash; two clusters drawn far apart are not necessarily more different than two drawn close together &mdash; whereas PCA's axes have a fixed, interpretable meaning (variance along a specific direction in the original feature space).</li>
        <li><b>There is no fixed transform</b> to place a new point into an existing t-SNE embedding the way <code>pca.transform(x_new)</code> places a new point into a PCA one; a fresh point requires rerunning the whole optimisation, while PCA's projection is a single matrix multiplication that applies to any new point immediately.</li>
    </ul>
    </p>
</details>

**25. A colleague at Aurora wants to (a) reduce the 8 account columns to a smaller set for a downstream model, and separately (b) produce a picture for a non-technical stakeholder showing that the account data forms recognisable groups. Which method fits each task, and why is neither one the right tool for the other's job?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>For (a), reach for <b>PCA</b>: it gives a fixed, interpretable transform &mdash; a small number of components ranked by variance explained, computed once and reusable on new data via a simple matrix multiplication (<code>pca.transform</code>) &mdash; exactly what a downstream model needs.</li>
        <li>For (b), <b>t-SNE</b> is typically better suited: it is built specifically to make genuinely separate clusters visually obvious, often more so than PCA's first two components manage, because that is exactly what it optimises for.</li>
        <li>Neither substitutes for the other's job: PCA's 2-D scatter is not built to maximise visual cluster separation the way t-SNE is, and t-SNE's embedding has no reusable transform and no meaningful inter-cluster distances, so it cannot serve as the fixed, quantitative reduction step (a) needs.</li>
    </ul>
    </p>
</details>